In [7]:
using LurCGT
using Telum
using LinearAlgebra


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [8]:
option = FermionSOptions(1, :U1, :SU2, nothing)
q = getLocalSpace(option);

# Reference & lazy-evaluation

To reduce memory usage, simple operations like scalar multiplication, permutation, and conjugation are evaluated lazily. For example, the code below does not allocate memory for tensor T2.

T2 is a struct conveying information: 1) reference to data of T1 and 2) scalar factor 2.

In [9]:
T1 = q.I
println(T1)
println(typeof(T1)) # Concrete TLArray
T2 = 2 * T1
println(T2)
println(typeof(T2)) # TLArrayView, have a reference to T1

2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	1.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	1.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	1.000000	√1
TLArray{Float64, 2, 2, 4, Tuple{Tuple{Int64}, Tuple{Int64}}, ProductSymm{Tuple{U1, SU{2}}}, 1, Array{Float64, 4}}
2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	2.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	2.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	2.000000	√1
TLArrayView{Float64, 2, 2, 4, Tuple{Tuple{Int64}, Tuple{Int64}}, ProductSymm{Tuple{U1, SU{2}}}, 1, Array{Float64, 4}, TLArray{Float64, 2, 2, 4, Tuple{Tuple{Int64}, Tuple{Int64}}, ProductSymm{Tuple{U1, SU{2}}}, 1, Array{Float64, 4}}}


If the first RMT of T1 is mutated, T2 is also affected.

In [10]:
T1.RMTs[1] = [4;;;;] 
println(T1)
println(T2) # The first RMT of T2 is also changed.

2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	4.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	1.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	1.000000	√1
2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	8.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	2.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	2.000000	√1


You can call the function 'to_concrete' to get an independent tensor.

In [14]:
q = getLocalSpace(option)
T1 = q.I
T2 = 2 * T1
println(T1)
T2_ = to_concrete(T2)
println(T2_)

2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	1.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	1.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	1.000000	√1
2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	2.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	2.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	2.000000	√1


In [16]:
T1.RMTs[1] = [4;;;;] 
println(T1)
println(T2_) # Not changed

2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	4.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	1.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	1.000000	√1
2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	2.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	2.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	2.000000	√1


Avoid mutation of tensor as far as possible. If it is inevitable, copy the original tensor first. 

In [17]:
q = getLocalSpace(option)
T1 = q.I
T2 = 2 * T1
T1_ = copy(T1)
println(T1_)
println(T2)

2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	1.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	1.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	1.000000	√1
2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	2.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	2.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	2.000000	√1


In [18]:
T1_.RMTs[1] = [4;;;;]
println(T1_)
println(T2) # Not changed

2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	4.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	1.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	1.000000	√1
2D TLArray, 2 symmetries [U1, SU2]  ["+", "-"]
  1.	1x1	| 1x1	[ -1 0 ; -1 0 ]	2.000000	√1
  2.	1x1	| 2x2	[  0 1 ;  0 1 ]	2.000000	√2
  3.	1x1	| 1x1	[  1 0 ;  1 0 ]	2.000000	√1


# Database management

LurCGT has a 3-level system to store CGT data.

1. Global database: Global storage so that every process can read. Only one process can write at a time.
2. Local database:  Local storage for a single process.
3. In-memory cache: Cache managed by a process for fast reuse.

LurCGT finds the CGT data (CG3, X-symbol, ...) from 3->2->1. If it does not exist, data is created and stored in 2 and 3.

By default, databases are placed in $HOME/.LurCGT_sqlite/ directory.


### In the local environment

The CGT data computed in LurCGT is stored on disk via SQlite. In the local environment, no additional settings are required. 

Databases are automatically created if they do not exist. Merging the local one with global and deleting local one occurs automatically when the process is terminated normally. Otherwise, local ones should be removed manually.

In [19]:
script = expanduser("~/.LurCGT_sqlite/")
run(`ls $script`)

global
local


Process(`ls /home/lurlurlur/.LurCGT_sqlite/`, ProcessExited(0))

In [20]:
script = expanduser("~/.LurCGT_sqlite/local/")
run(`ls $script`)

Sp4
Sp6
SU2
SU3


Process(`ls /home/lurlurlur/.LurCGT_sqlite/local/`, ProcessExited(0))

In [21]:
script = expanduser("~/.LurCGT_sqlite/local/SU2")
run(`ls $script`)

lurlurlur-MS-7C94_pid17002.db
lurlurlur-MS-7C94_pid17002.db-shm
lurlurlur-MS-7C94_pid17002.db-wal
lurlurlur-MS-7C94_pid20625.db
lurlurlur-MS-7C94_pid20625.db-shm
lurlurlur-MS-7C94_pid20625.db-wal
lurlurlur-MS-7C94_pid26611.db
lurlurlur-MS-7C94_pid26611.db-shm
lurlurlur-MS-7C94_pid26611.db-wal
lurlurlur-MS-7C94_pid28478.db
lurlurlur-MS-7C94_pid28478.db-shm
lurlurlur-MS-7C94_pid28478.db-wal
lurlurlur-MS-7C94_pid32003.db
lurlurlur-MS-7C94_pid32003.db-shm
lurlurlur-MS-7C94_pid32003.db-wal
lurlurlur-MS-7C94_pid32174.db
lurlurlur-MS-7C94_pid32174.db-shm
lurlurlur-MS-7C94_pid32174.db-wal
lurlurlur-MS-7C94_pid32750.db
lurlurlur-MS-7C94_pid32750.db-shm
lurlurlur-MS-7C94_pid32750.db-wal
lurlurlur-MS-7C94_pid36229.db
lurlurlur-MS-7C94_pid36229.db-shm
lurlurlur-MS-7C94_pid36229.db-wal
lurlurlur-MS-7C94_pid36521.db
lurlurlur-MS-7C94_pid36521.db-shm
lurlurlur-MS-7C94_pid36521.db-wal


Process(`ls /home/lurlurlur/.LurCGT_sqlite/local/SU2`, ProcessExited(0))

In [ ]:
script = expanduser("~/.LurCGT_sqlite/global")
run(`ls $script`)

SU2.db
SU2.db-shm
SU2.db-wal
SU3.db
SU3.db-shm
SU3.db-wal
locks


Process(`ls '~/.LurCGT_sqlite/global'`, ProcessExited(0))

### In server

!!!This part can be changed later.

To use the LurCGT database system efficiently in a cluster, 4 environment variables are required.

<span style="color: red;">LURCGT_RUN_MODE=server</span>:
The variables below are activated when this is set to 'server'.

<span style="color: red;">LURCGT_LOCALDB_DIR_NODE</span>:
The location of the local database inside the computing node.

<span style="color: red;">LURCGT_GLOBALDB_DIR_NODE</span>:
The location of the global database inside the computing node.

<span style="color: red;">LURCGT_GLOBALDB_DIR</span>:
The location of the global database inside the head node.

The variables ending with '_NODE' are recommended to be inside the local scratch of the computing node for performance.

When the computation is requested through the Slurm command,

1. The global database in the head node (in LURCGT_GLOBALDB_DIR) is copied to the LURCGT_GLOBALDB_DIR_NODE directory.

2. A local database is created inside the node, and computation takes place using the local and global DBs in the node. 

3. After finishing, the local DB is merged into the global one in the head node.

4. DBs in the computing node are deleted when terminated normally.

The reason for copying the global DB is for performance. Getting data from the head node via NFS(network file system) is extremely slow.